# 第一回: 入試問題を解く (その1) 記述する

## DNCLによる出題を自然言語として分析する

In [ ]:
# 図2 ２人目以降の来訪者の待ち時間を求めるプログラム
dncl_example = '''(01) taiken = 3
(02) Touchaku = [0, 3, 4, 10, 11, 12]
(03) kyakusu = 要素数(Touchaku)
(04) Kaishi = [0, 0, 0, 0, 0, 0]
(05) Shuryou = [0, 0, 0, 0, 0, 0]
(06) Shuryou[1] = taiken
(07) i を 2 から kyakusu まで 1 ずつ増やしながら繰り返す:
(08)     Kaishi[i] = 最大値( カ , キ )
(09)     Shuryou[i] = ク
(10)     表示する(i, "人目の待ち時間:", ケ - コ, "分間")
'''

In [ ]:
# 図3 最長待ち時間が10分間未満となる体験時間を調べるプログラム
dncl_example = '''(01) Touchaku = [0, 3, 4, 10, 11, 12]
(02) kyakusu = 要素数(Touchaku)
(03) taiken を 1 から 15 まで 1 ずつ増やしながら繰り返す:
(04)     Kaishi = [0, 0, 0, 0, 0, 0]
(05)     Shuryou = [0, 0, 0, 0, 0, 0]
(06)     Shuryou[1] = taiken
(07)     i を 2 から kyakusu まで 1 ずつ増やしながら繰り返す:
(08)         Kaishi[i] = 最大値( カ , キ )
(09)         Shuryou[i] = ク
(10)     saichou = 0
(11)     i を 1 から kyakusu まで 1 ずつ増やしながら繰り返す:
(12)         saichou = 最大値( サ , ケ - コ )
(13)     もし シ ならば:
(14)         表示する("体験時間", taiken, "分間: ", 最長待ち時間", saichou, "分間") 
'''
# (04)〜(09)行目の処理は図2のプログラムの(04)〜(09)行目の処理と同じである

In [ ]:
# 図3 最長待ち時間が10分間未満となる体験時間を調べるプログラム
dncl_example = '''(01) Touchaku = [0, 3, 4, 10, 11, 12]
(02) kyakusu = 要素数(Touchaku)
(03) taiken = 1, saichou = 0
(04) (taiken <= 15) and (saichou < 10) の間繰り返す:
(05)     Kaishi = [0, 0, 0, 0, 0, 0]
(06)     Shuryou = [0, 0, 0, 0, 0, 0]
(07)     Shuryou[1] = taiken
(08)     i を 2 から kyakusu まで 1 ずつ増やしながら繰り返す:
(09)         Kaishi[i] = 最大値( カ , キ )
(10)         Shuryou[i] = ク
(11)     saichou = 0
(12)     i を 1 から kyakusu まで 1 ずつ増やしながら繰り返す:
(13)         saichou = 最大値( サ , ケ - コ )
(14)     もし シ ならば:
(15)         表示する("体験時間", taiken, "分間: ", 最長待ち時間", saichou, "分間") 
(16)     taiken = taiken + 1
'''
# (04)〜(09)行目の処理は図2のプログラムの(04)〜(09)行目の処理と同じである

### 1.4.1 行番号を取り除く

In [ ]:
import re

def strip_lineno(s: str) -> str:
    # Robustly remove a leading parenthesized prefix like '(01)' while preserving indentation.
    # This handles cases like '(13)     もし ...' by keeping the spaces after the closing paren.
    m = re.match(r'^(\s*)\(([^)]*)\)(\s*)(.*)$', s)
    if m:
        leading_spaces = m.group(1)  # spaces before '(' (often empty)
        spaces_after = m.group(3)    # spaces after ')' which encode indentation
        rest = m.group(4)            # remaining content
        return leading_spaces + spaces_after + rest.rstrip('\n')
    # If no parenthesized prefix, return line stripped of trailing newline only
    return s.rstrip('\n')

In [ ]:
example_lines = dncl_example.splitlines()
example_lines

In [ ]:
for l in example_lines:
    print(strip_lineno(l))

### 1.4.2 分かち書き (字句解析)

```{seealso}
* [Lexical analysis - Wikipedia](https://en.wikipedia.org/wiki/Lexical_analysis)
```

In [ ]:
def leading_spaces(s: str) -> int:
    return len(s) - len(s.lstrip(' '))


def tok_types(line: str):
    # normalize fullwidth digits to ASCII so they are recognized as numbers
    fw_digits = str.maketrans('０１２３４５６７８９', '0123456789')
    line = line.translate(fw_digits)
    tokens = []
    i = 0
    # known function names in DNCL (Japanese)
    funcs = ['要素数', '最大値', '表示する']
    L = len(line)
    while i < L:
        c = line[i]
        if c.isspace():
            i += 1
            continue
        matched = False
        for f in funcs:
            if line.startswith(f, i):
                tokens.append('FUNC')
                i += len(f)
                matched = True
                break
        if matched:
            continue
        # string (handle escaped quotes)
        if line[i] in '"\'':
            q = line[i]
            j = i+1
            while j < L:
                if line[j] == '\\' and j+1 < L:
                    j += 2
                    continue
                if line[j] == q:
                    break
                j += 1
            if j >= L:
                j = L-1
            tokens.append('STRING')
            i = j+1
            continue
        # number (ASCII digits)
        m = re.match(r'\d+', line[i:])
        if m:
            tokens.append('NUMBER')
            i += len(m.group(0))
            continue
        # comparison operators
        if line.startswith('<=', i):
            tokens.append('LE')
            i += 2
            continue
        if line.startswith('>=', i):
            tokens.append('GE')
            i += 2
            continue
        if line[i] == '<':
            tokens.append('LT')
            i += 1
            continue
        if line[i] == '>':
            tokens.append('GT')
            i += 1
            continue
        # logical words (and/or)
        m2 = re.match(r'(and|or)\b', line[i:], re.I)
        if m2:
            w = m2.group(1).lower()
            tokens.append('AND' if w=='and' else 'OR')
            i += len(m2.group(0))
            continue
        # punctuation
        if line[i] in '[],()=:+-*':
            tokens.append(line[i])
            i += 1
            continue
        # identifier: ASCII IDs or any run of non-separator characters (including Japanese),
        # but ensure it does NOT start with a digit so numbers are not captured as ID
        m_id = re.match(r"[A-Za-z_][A-Za-z0-9_]*|(?!\d)[^\s,=+\-\*()\[\]<>:,]+", line[i:])
        if m_id:
            tokens.append('ID')
            i += len(m_id.group(0))
            continue
        # fallback: single character token
        tokens.append(line[i])
        i += 1
    return tokens


def tokenize_lines_with_indent(lines):
    results = []
    indent_stack = [0]
    first_real_line_seen = False
    last_significant_toks = None
    last_header_indent = None
    expect_indent_on_next = False
    for idx, raw in enumerate(lines, start=1):
        s = strip_lineno(raw)
        if not s.strip() or s.strip().startswith('#'):
            results.append((idx, ['SKIP']))
            continue
        cur = leading_spaces(s)
        toks = []
        # Decide whether this increase of indent is logical: only after a header-like line
        prev_was_header = False
        if last_significant_toks:
            prev_was_header = (last_significant_toks and (last_significant_toks[-1] == ':' or last_significant_toks[0] in ('LOOP_HEADER', 'REPEAT_HEADER', 'IF_HEADER')))
        content = s.lstrip(' ')
        content_stripped = content.rstrip()
        is_header_line = False
        # Emit INDENT only when indent increases AND previous line was a header (or repetition) or explicit expect flag
        if (cur > indent_stack[-1] and first_real_line_seen and (prev_was_header or (last_header_indent is not None and cur > last_header_indent))) or expect_indent_on_next:
            indent_stack.append(cur)
            toks.append('INDENT')
            expect_indent_on_next = False
        else:
            while cur < indent_stack[-1]:
                indent_stack.pop()
                toks.append('DEDENT')
        first_real_line_seen = True
        # Determine header or normal line: detect three header patterns and treat them uniformly as headers
        if content_stripped.endswith(':'):
            # 1) 'の間繰り返す:' pattern (condition の間繰り返す:)
            if 'の間繰り返す' in content_stripped:
                i_pos = content_stripped.find('の間繰り返す')
                cond = content_stripped[:i_pos].strip()
                if cond:
                    toks.extend(tok_types(cond))
                toks.append('REPEAT_HEADER')
                toks.append(':')
                is_header_line = True
            # 2) 'ながら繰り返す:' pattern (loop header like '増やしながら繰り返す:')
            elif 'ながら繰り返す' in content_stripped:
                i_pos = content_stripped.find('ながら繰り返す')
                head = content_stripped[:i_pos].strip()
                if head:
                    toks.extend(tok_types(head))
                toks.append('LOOP_HEADER')
                toks.append(':')
                is_header_line = True
            # 3) 'もし ... ならば:' pattern (if header)
            else:
                m_if = re.match(r'^もし\s+(.*?)\s+ならば\s*:$', content_stripped)
                if m_if:
                    cond = m_if.group(1).strip()
                    if cond:
                        toks.extend(tok_types(cond))
                    toks.append('IF_HEADER')
                    toks.append(':')
                    is_header_line = True
                else:
                    # fallback: generic loop/header token
                    toks.append('LOOP_HEADER')
                    toks.append(':')
                    is_header_line = True
        else:
            toks.extend(tok_types(content_stripped))
        # record header indent for next line decision
        if is_header_line:
            last_header_indent = cur
            expect_indent_on_next = True
        else:
            last_header_indent = None
        results.append((idx, toks))
        if toks != ['SKIP']:
            last_significant_toks = toks
    while len(indent_stack) > 1:
        indent_stack.pop()
        results.append(('EOF', ['DEDENT']))
    return results

In [ ]:
# Tokenize with INDENT/DEDENT info
tokenized = tokenize_lines_with_indent(example_lines)
print('=== Pass1: per-line tokens ===')
for idx, toks in tokenized:
    print(*toks)

### 1.4.3 nltkによる構文解析

```{seealso}
* (2001&ndash;) [Natural Language Toolkit - Wikipedia](https://en.wikipedia.org/wiki/Natural_Language_Toolkit)
```

In [ ]:
import nltk
from nltk import CFG, ChartParser

# Line-level grammar for per-line parsing (headers handled by grammar)
line_grammar = r'''
S -> ASSIGN | FUNC_CALL | LOOP_STMT | IF_STMT
LOOP_STMT -> 'LOOP_HEADER' ':' | COND 'REPEAT_HEADER' ':'
IF_STMT -> 'IF_HEADER' ':' | COND 'IF_HEADER' ':'
COND -> EXPR | EXPR 'AND' EXPR | EXPR 'OR' EXPR
ASSIGN -> 'ID' '=' EXPR | 'ID' '[' 'NUMBER' ']' '=' EXPR | 'ID' '[' 'ID' ']' '=' EXPR
EXPR -> 'NUMBER' | 'ID' | FUNC_CALL | EXPR '+' EXPR | EXPR '-' EXPR | EXPR '*' EXPR | ARRAY
ARRAY -> '[' ELEM_LIST ']'
ELEM_LIST -> ELEMENT | ELEMENT ',' ELEM_LIST
ELEMENT -> EXPR
FUNC_CALL -> 'FUNC' '(' ARGS ')' | 'FUNC' '(' ')'
ARGS -> ARG | ARG ',' ARGS
ARG -> EXPR | 'STRING'
'''

try:
    cfg_line = CFG.fromstring(line_grammar)
    parser_line = ChartParser(cfg_line)
    print('\n=== Pass2: per-line parse trees ===')
    for idx, toks in tokenized:
        if idx == 'EOF':
            continue
        if toks == ['SKIP']:
            continue
        # remove layout tokens for line parsing
        line_tokens = [t for t in toks if t not in ('INDENT', 'DEDENT')]
        # remove optional leading lineno tokens like '(' 'NUMBER' ')'
        if len(line_tokens) >= 3 and line_tokens[0] == '(' and line_tokens[1] == 'NUMBER' and line_tokens[2] == ')':
            line_tokens = line_tokens[3:]
        try:
            trees = list(parser_line.parse(line_tokens))
            if trees:
                for t in trees:
                    print(f'Line {idx} ->', t)
            else:
                print(f'Line {idx} -> No parse')
        except Exception as e:
            print(f'Line {idx} parse error:', e)
except Exception as e:
    print('Line parser setup error:', e)

```{attention}
* nltkでは、複数行にまたがる複合命令文 (compaund statement) を解析できない
    - DNCLによる出題を自然言語として処理しようとしても解答できない可能性が高い
```